# Notebook 09b: equivariance-coupled retrain (Idea 9, arm 2 scaffold)

## STATUS: PARTLY SUPERSEDED. Read this box before reading anything else.

This notebook is **arm 2's plumbing scaffold, not arm 2's result**, and two specific things in it have
been superseded by later work. Both are kept in place, clearly labelled, because the mistakes are
instructive.

1. **Every number this notebook computes is SMOKE-SCALE SYNTHETIC.** It always builds a hand-authored
   synthetic cohort (section 3) and always trains a tiny model for a handful of epochs (section 5),
   regardless of what `GAVD_MODE` says. In the executed run below, `GAVD_MODE` was `real`, so the cell
   outputs print `artifact mode: real`. That label refers ONLY to which directory the bundle is written
   to. The data is synthetic and the training scale is smoke, which is why the bundle is labelled by
   `data_source` and `training_scale` instead. Its ladder result, **D0 mean R-squared -0.320 with standard
   deviation 0.036 against E1 mean -0.294, an effect of 0.025, credited false**, is PLUMBING VALIDATION
   ONLY. It is not evidence about any checkpoint, and it must never be quoted as an arm 2 finding.
2. **The equivariance term this notebook defines is DEFECTIVE, and section 8 explains exactly how.** The
   absolute form `L_equiv = mean( ( s(enc(Mx)) + s(enc(x)) )^2 )` with a TRAINABLE head `s` is
   scale-degenerate: it can be driven toward zero by shrinking the head's own output instead of by
   changing the encoder. Section 8 also marks the recipe for pasting this term into notebook 04 as
   SUPERSEDED, DO NOT FOLLOW AS WRITTEN.

**Where arm 2 actually happened, and what it found.** The real run is the four-notebook series
`new_nb_09_00_methodology_and_contract`, `new_nb_09_01_mechanism_and_smoke_validation`,
`new_nb_09_02_real_multiseed_equivariant_training`, and
`new_nb_09_03_evaluation_results_discussion`. It used a repaired, scale-invariant term and a label-free
endpoint, and its preregistered verdict is **NO CREDIT**. Section 8 reads that verdict out of the series'
own bundle rather than restating it.

## What this notebook is still good for

*The question it was written to answer.* Arm 1 (`nb_09a`) never touches the encoder: it asks whether an
antisymmetry-constrained READOUT beats a binding bar on frozen features, and its verdict is
`ARTIFACT (side-agnostic nuisance control fired)`. Arm 1 could have failed for a reason arm 1 cannot
test, namely that the encoder was never ASKED to respect the mirror during training. Arm 2 is the only
arm that can change that, by adding a label-free equivariance term to the training loss and retraining.

*What this notebook contributes to that.* It is the scaffold that demonstrates the machinery: the D0
versus E1 ablation ladder shape, per-rung fingerprinting so no rung can be confused with the baseline,
the collapse monitors, and the credit-rule arithmetic. It also contains, unintentionally, a worked
demonstration of the failure mode that the real run had to repair.

*The one new loss term, stated once so the rest of the notebook can refer to it.* With the antisymmetric
head `s` from `nb_09a` and the anatomical mirror `M` on the RAW skeleton (which reflects the body AND
swaps left and right landmark identities, not just one of the two),

`L_equiv = mean( ( s(encoder(Mx)) + s(encoder(x)) )^2 )`.

Both the original and the mirrored skeleton are run THROUGH the view encoder before the head reads them.
Section 2 explains why "through the encoder" is not a detail but the whole point. The term uses NO labels.
A label-supervised axis term was deliberately NOT added, because with roughly seven lateralized source
videos a supervised axis would be a vacuous transductive win.

*Why the experiment is an ablation ladder rather than a single run.* `D0` reproduces the recipe with the
equivariance weight set to 0, and `E1` is `D0` plus `L_equiv`. Running both across seeds and crediting the
effect only when it exceeds `D0`'s seed-to-seed spread is a TRAJECTORY control: it asks whether the effect
is larger than the noise of rerunning the same recipe. Note what it is not: it is not a source-variation
claim, because the seeds vary the optimisation, not the cohort.

*Standing caveats.* All results are transductive, the source video is the independent unit of evidence,
and folder labels are dataset annotations, not clinical diagnoses. On top of those, the caveat specific to
this notebook: the numbers here are synthetic and smoke scale, so they are plumbing checks and nothing
else.


## 0. Environment, mode, and the run knobs

**Step 1 of 9.**

*What we are about to do.* Resolve the project root the way notebooks 04 through 06 and `nb_05a` do, read
the arm 2 knobs from the environment, and print the total loss the ladder will optimise.

*Why the knobs are environment-driven with defaults.* So a reader can change the experiment without
editing code, and so the exact configuration is printed into the output rather than being implicit. The
knobs, defined before they are used:

- `IDEA9_EQUIV_WEIGHT`, the weight on `L_equiv`, default **0.02**, deliberately small relative to the
  main terms so the equivariance term nudges rather than dominates.
- `IDEA9_SEEDS`, the seed list for the D0 versus E1 ladder, default `0,1,2`.
- `IDEA9_HEAD_OUT_DIM`, the head's output width `m`, default 4, matching `nb_09a`.
- The existing notebook 04 weights, reprinted so the ladder's loss is auditable: VICReg weight 0.05, group
  weight 0.25, group margin 1.0, mask fraction 0.60.

*Terminology, fixed by the project and used consistently below.* **VICReg** is only the label-free
invariance, variance-hinge, and covariance regulariser computed on projected student features. The **group
loss** is a separate LABEL-AWARE term, equal to within-condition compactness plus a centroid-margin
penalty. These are two different things and the printed loss line lists them separately.

*What to look at in the output.* `mode : real`, and then the loss line
`jepa + 0.05*vicreg + 0.25*group + 0.02*L_equiv`.

*The trap in that first line, which is the reason this notebook was relabelled.* `mode : real` does NOT
mean this notebook ran on real data at real scale. `MODE` is only the artifact directory. Section 5 makes
this explicit and asserts it, and because `GAVD_MODE` was `real` here, the `SMOKE MODE` banner did not
print. A reader glancing at this output could easily mistake what follows for a real result. It is not
one.


In [1]:
from pathlib import Path
import os, math, json, hashlib, copy, warnings

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


def find_project_root(start=None):
    env_root = os.getenv("ALEXPOSE_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
TUTORIAL_DIR = PROJECT_ROOT / "experiments" / "sjepa" / "gavd6-pm"
try:
    from dotenv import load_dotenv
    load_dotenv(TUTORIAL_DIR / ".env", override=False)
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except Exception:
    pass

ARTIFACT_ROOT = Path(os.getenv("GAVD_ARTIFACT_DIR", TUTORIAL_DIR / "work" / "artifacts")).expanduser()
REQUESTED_MODE = os.getenv("GAVD_MODE", "smoke").strip().lower()
if REQUESTED_MODE not in {"smoke", "real"}:
    raise ValueError("GAVD_MODE must be smoke or real")
MODE = REQUESTED_MODE

CONDITIONS = ["normal", "parkinsons", "stroke", "myopathic", "cerebralpalsy"]
CURRICULUM = [
    {"stage": 0, "name": "normal_only", "add": "normal", "conditions": ["normal"]},
    {"stage": 1, "name": "add_parkinsons", "add": "parkinsons", "conditions": CONDITIONS[:2]},
    {"stage": 2, "name": "add_stroke", "add": "stroke", "conditions": CONDITIONS[:3]},
    {"stage": 3, "name": "add_myopathic", "add": "myopathic", "conditions": CONDITIONS[:4]},
    {"stage": 4, "name": "add_cerebralpalsy", "add": "cerebralpalsy", "conditions": CONDITIONS[:5]},
]
MASK_KEYPOINTS = [11, 12, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
LEFT_RIGHT_PAIRS = [(11, 12), (23, 24), (25, 26), (27, 28), (29, 30), (31, 32)]

# ---- Arm-2 knobs (env-driven, safe defaults) ----
EQUIV_WEIGHT = float(os.getenv("IDEA9_EQUIV_WEIGHT", "0.02"))
SEEDS = [int(s) for s in os.getenv("IDEA9_SEEDS", "0,1,2").split(",") if s.strip() != ""]
HEAD_OUT_DIM = int(os.getenv("IDEA9_HEAD_OUT_DIM", "4"))
VICREG_WEIGHT = float(os.getenv("SJEPA_VICREG_WEIGHT", "0.05"))
GROUP_WEIGHT = float(os.getenv("SJEPA_GROUP_WEIGHT", "0.25"))
GROUP_MARGIN = float(os.getenv("SJEPA_GROUP_MARGIN", "1.0"))
MASK_FRACTION = float(os.getenv("SJEPA_MASK_FRACTION", "0.60"))

OUT_DIR = ARTIFACT_ROOT / MODE
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"mode         : {MODE}")
print(f"EQUIV_WEIGHT : {EQUIV_WEIGHT}")
print(f"SEEDS        : {SEEDS}")
print(f"total loss   : jepa + {VICREG_WEIGHT}*vicreg + {GROUP_WEIGHT}*group + {EQUIV_WEIGHT}*L_equiv")
if MODE == "smoke":
    print("SMOKE MODE: synthetic cohort, tiny model, few epochs. Numbers are plumbing checks only.")

PROJECT_ROOT : /Users/pmui/dev/alexpose
mode         : real
EQUIV_WEIGHT : 0.02
SEEDS        : [0, 1, 2]
total loss   : jepa + 0.05*vicreg + 0.25*group + 0.02*L_equiv


## 1. Model classes and loss functions (reused verbatim from notebook 04)

**Step 2 of 9.**

*What we are about to do.* Paste notebook 04's `SkeletonPatchEncoder`, `SkeletonPredictor`, `SJEPAGait`
(with `update_target` and `update_center`), `sjepa_cross_entropy`, `cosine_ema`, and `geometric_view`
exactly as notebook 04 defines them.

*Why verbatim, and what would go wrong otherwise.* Two reasons, both practical. First, a retrained
checkpoint must be drop-in loadable by the `nb_09a` instrument, which matches `state_dict` keys by name,
so the class definitions have to correspond key for key. Second, the ladder is only interpretable if D0
reproduces the BASELINE recipe; if any loss term drifted, then the D0-versus-E1 difference would mix the
equivariance term with an unrelated change and the trajectory control would be meaningless.

*One augmentation detail that is load-bearing for this experiment.* `geometric_view` keeps
`flip_probability = 0.0`. Left and right identity is exactly what this whole line of work is about, so
randomly mirroring training views would teach the encoder to IGNORE the distinction that arm 2 is trying
to install. The flip branch is present in the code but is never taken at this probability.

*What to look at in the output.* The single line
`model + loss + augmentation defined (verbatim from notebook 04).` Nothing is measured here.


In [2]:
try:
    import torch
    from torch import nn
    from torch.nn import functional as F
    HAVE_TORCH = True
except Exception as exc:  # pragma: no cover
    HAVE_TORCH = False
    raise RuntimeError(f"Arm 2 requires PyTorch: {exc}")


class SkeletonPatchEncoder(nn.Module):
    def __init__(self, frames=64, joints=33, coordinate_dim=3, segment_length=4,
                 embed_dim=64, depth=2, heads=4, dropout=0.0):
        super().__init__()
        if frames % segment_length:
            raise ValueError("frames must be divisible by segment_length")
        self.frames, self.joints, self.coordinate_dim = frames, joints, coordinate_dim
        self.segment_length, self.embed_dim = segment_length, embed_dim
        self.segments = frames // segment_length
        self.patch_embed = nn.Linear(segment_length * coordinate_dim, embed_dim)
        self.time_pos = nn.Parameter(torch.randn(self.segments, embed_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, embed_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=heads, dim_feedforward=embed_dim * 4,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

    def patchify(self, x):
        batch, frames, joints, channels = x.shape
        if (frames, joints, channels) != (self.frames, self.joints, self.coordinate_dim):
            raise ValueError(f"bad shape {x.shape}")
        patches = x.reshape(batch, self.segments, self.segment_length, joints, channels)
        return patches.permute(0, 1, 3, 2, 4).contiguous().flatten(3)

    def positioned_tokens(self, x):
        tokens = self.patch_embed(self.patchify(x))
        return tokens + self.time_pos[None, :, None, :] + self.joint_pos[None, None, :, :]

    def forward(self, x, keep_mask=None):
        tokens = self.positioned_tokens(x)
        batch = len(tokens)
        flat = tokens.reshape(batch, self.segments * self.joints, self.embed_dim)
        if keep_mask is not None:
            keep_mask = keep_mask.reshape(batch, -1)
            kept = keep_mask.sum(dim=1)
            if not torch.equal(kept, kept[:1].expand_as(kept)):
                raise ValueError("Each sample must keep the same number of tokens")
            flat = flat[keep_mask].reshape(batch, int(kept[0]), self.embed_dim)
        return self.norm(self.blocks(flat))


class SkeletonPredictor(nn.Module):
    def __init__(self, segments, joints, encoder_dim=64, predictor_dim=64, depth=2, heads=4, dropout=0.0):
        super().__init__()
        self.segments, self.joints = segments, joints
        self.encoder_to_predictor = nn.Linear(encoder_dim, predictor_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.normal_(self.mask_token, std=0.02)
        self.time_pos = nn.Parameter(torch.randn(segments, predictor_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, predictor_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=predictor_dim, nhead=heads, dim_feedforward=predictor_dim * 4,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(predictor_dim)
        self.output = nn.Linear(predictor_dim, encoder_dim)

    def forward(self, visible_features, target_mask):
        batch = len(visible_features)
        target_mask = target_mask.reshape(batch, self.segments * self.joints)
        visible_mask = ~target_mask
        visible = self.encoder_to_predictor(visible_features)
        full = self.mask_token.expand(batch, self.segments * self.joints, -1).clone()
        full[visible_mask] = visible.reshape(-1, visible.shape[-1])
        positions = (self.time_pos[:, None, :] + self.joint_pos[None, :, :]).reshape(
            1, self.segments * self.joints, -1)
        full = full + positions
        predicted = self.output(self.norm(self.blocks(full)))
        return predicted[target_mask].reshape(batch, -1, predicted.shape[-1])


class SJEPAGait(nn.Module):
    def __init__(self, frames=64, joints=33, coordinate_dim=3, segment_length=4,
                 embed_dim=64, encoder_depth=2, predictor_depth=2, heads=4):
        super().__init__()
        self.view_encoder = SkeletonPatchEncoder(
            frames, joints, coordinate_dim, segment_length, embed_dim, encoder_depth, heads)
        self.target_encoder = copy.deepcopy(self.view_encoder)
        for parameter in self.target_encoder.parameters():
            parameter.requires_grad_(False)
        self.predictor = SkeletonPredictor(
            self.view_encoder.segments, joints, embed_dim, embed_dim, predictor_depth, heads)
        self.register_buffer("target_center", torch.zeros(embed_dim))

    def forward(self, view, target, target_mask):
        visible_features = self.view_encoder(view, keep_mask=~target_mask)
        predicted = self.predictor(visible_features, target_mask)
        with torch.no_grad():
            all_targets = self.target_encoder(target)
            flat_mask = target_mask.reshape(len(target), -1)
            selected = all_targets[flat_mask].reshape(len(target), -1, all_targets.shape[-1])
        return predicted, selected

    @torch.no_grad()
    def update_target(self, momentum):
        for tp, vp in zip(self.target_encoder.parameters(), self.view_encoder.parameters()):
            tp.mul_(momentum).add_(vp, alpha=1.0 - momentum)

    @torch.no_grad()
    def update_center(self, targets, beta=0.9):
        self.target_center.mul_(beta).add_(targets.mean(dim=(0, 1)), alpha=1.0 - beta)


def sjepa_cross_entropy(predicted, targets, center, predictor_temperature=0.10, target_temperature=0.06):
    target_prob = torch.softmax((targets - center[None, None, :]) / target_temperature, dim=-1).detach()
    prediction_log_prob = torch.log_softmax(predicted / predictor_temperature, dim=-1)
    return -(target_prob * prediction_log_prob).sum(dim=-1).mean()


def cosine_ema(step, total_steps, start=0.996, end=1.0):
    progress = min(max(step / max(total_steps - 1, 1), 0.0), 1.0)
    return end - (end - start) * (math.cos(math.pi * progress) + 1.0) / 2.0


FULL_LR_PAIRS = [(1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16),
                 (17, 18), (19, 20), (21, 22), (23, 24), (25, 26), (27, 28), (29, 30), (31, 32)]


def geometric_view(x, max_degrees=8.0, translate=0.03, flip_probability=0.0):
    '''Verbatim from nb04: y-axis rotation (mixes x,z) + small translation; flip OFF by default.'''
    view = x.clone()
    present = view.abs().sum(dim=-1) > 1e-8
    batch = len(view)
    angles = (torch.rand(batch, device=x.device) * 2.0 - 1.0) * math.radians(max_degrees)
    cosine, sine = torch.cos(angles), torch.sin(angles)
    ox, oz = view[..., 0].clone(), view[..., 2].clone()
    view[..., 0] = cosine[:, None, None] * ox + sine[:, None, None] * oz
    view[..., 2] = -sine[:, None, None] * ox + cosine[:, None, None] * oz
    offsets = (torch.rand(batch, 1, 1, 2, device=x.device) * 2.0 - 1.0) * translate
    view[..., :2] += offsets
    if flip_probability > 0:
        flip = torch.rand(batch, device=x.device) < flip_probability
        for bi in torch.where(flip)[0].tolist():
            view[bi, ..., 0] *= -1.0
            original = view[bi].clone()
            for left, right in FULL_LR_PAIRS:
                view[bi, :, left] = original[:, right]
                view[bi, :, right] = original[:, left]
    view = view.masked_fill(~present[..., None], 0.0)
    return view


print("model + loss + augmentation defined (verbatim from notebook 04).")

model + loss + augmentation defined (verbatim from notebook 04).


## 2. The antisymmetric head and the label-free equivariance loss

**Step 3 of 9. This section contains the notebook's central correctness lesson AND its central defect, so
read both halves.**

*What we are about to build.* The same head as `nb_09a`, `s = sum over pairs k of ( f(L_k) - f(R_k) )`
with a shared `f = Linear(D,32) -> GELU -> Linear(32,m)` and no `l + r` term. The one difference from arm
1: here `f`'s parameters ARE trainable and join the optimiser, so the term can shape the encoder and the
head together.

### The lesson: the loss must go THROUGH the encoder, not around it

*The mistake a first draft made.* The obvious-looking loss is to swap the head's own input tokens and
penalise `s(swap of tokens) + s(tokens)`. That is **an algebraic no-op**. The head is antisymmetric under
a swap of its own inputs BY CONSTRUCTION, so that quantity is identically zero for every input and every
parameter value. Its loss is exactly **0.0** and its gradient into the head is exactly **0.0**. It trains
nothing at all. `new_nb_09_01` measured precisely this on synthetic fixtures, confirming a wiring slope of
-1.0000000000000007 with zero loss and zero gradient.

*Why running the mirror through the encoder fixes that.* The ENCODER is not equivariant by construction.
So reflecting the RAW skeleton anatomically, running BOTH the original and the mirrored skeleton through
the VIEW encoder, and penalising `s(enc(Mx)) + s(enc(x))` is zero only when the encoder has learned to
represent the mirrored body as the sign-flip of the original along the head's axis. That is a real,
non-trivial constraint on the encoder's weights, and it is still label-free.

*The general lesson, which is worth more than this notebook's numbers.* A regulariser that penalises a
quantity your architecture already guarantees will report a beautiful loss curve and change nothing. Always
verify that a new term has nonzero gradient into the parameters you intend to move.

### The defect: this ABSOLUTE form is scale-degenerate, and it is superseded

*What is wrong with it.* Because `s` is trainable and the residual is ABSOLUTE (an unnormalised squared
difference), the objective has a degenerate solution. The head can shrink its own output toward zero,
which drives `L_equiv` down while the encoder remains exactly as mirror-blind as it started. The loss curve
looks like success. Nothing was learned.

*How badly, measured.* On synthetic fixtures in `new_nb_09_01`, this absolute form drove its own term down
by a factor of **184.16** while the head's output scale shrank by a factor of **4.77**, and moved the
label-free mirror residual **rho** by only **0.0096** against a gate of **0.0492**. Those are synthetic
fixture numbers, not gait results. rho is scaled so that **0 is mirror equivariant and 4 is mirror
blind**.

*The repair, which the real run uses instead.* Divide the residual by the mean squared norm of the two
branches, making the term scale-invariant so that shrinking the head scales numerator and denominator
together and buys nothing. The real ladder went further and used a `parameter_free` variant, the same ratio
with `s` replaced by a FIXED antisymmetric contraction, so there are no head parameters left to shrink.
See `new_nb_09_01` for the bakeoff and `new_nb_09_02` for the run.

*What the two inline guardrails in the cell below do and do not prove.*

- Guardrail (a) checks that the head negates under a swap of its OWN input. This is a valid wiring
  self-check of the head, and it is explicitly NOT the training loss.
- Guardrail (b) checks that the real, through-the-encoder loss is strictly positive with nonzero gradients
  into BOTH the head and the encoder. Note what it is measured on: a deliberately non-equivariant random
  `Linear` stand-in, not the real encoder. So it establishes that the term is not a no-op. It does NOT
  establish that the term is well-posed, and indeed it is not: guardrail (b) passes for a loss that is
  still satisfiable by shrinking the head. That gap is exactly why the mechanism validation in
  `new_nb_09_01` was necessary.

*What to look at in the output.*

- `guardrail (a): head own-input swap slope = -1.000000`, exact by construction.
- `guardrail (b): L_equiv=1.6171e-01 (> 0) head|grad|=1.586e+00 encoder|grad|=8.008e-01`, so the term
  reaches both.
- A `UserWarning` about converting a tensor with `requires_grad=True` to a scalar, raised inside the
  assertion. It is cosmetic and does not affect the values.


In [3]:
FULL_LR_PAIRS_MIRROR = FULL_LR_PAIRS  # 16-pair anatomical mirror, same as nb04's geometric_view flip


def anatomical_mirror_coords(coords):
    '''Reflect the RAW skeleton: negate the sideways (x) coordinate and swap each left/right landmark.
    coords: [B, FRAMES, 33, 3] tensor. Returns the mirrored tensor (differentiable passthrough).'''
    m = coords.clone()
    m[..., 0] = -m[..., 0]
    idx = list(range(33))
    for li, ri in FULL_LR_PAIRS_MIRROR:
        idx[li], idx[ri] = ri, li
    return m[:, :, idx, :]


class AntisymmetricHead(nn.Module):
    '''s = sum_k ( f(L_k) - f(R_k) ); f shared across joints/sides; difference only. Trainable in Arm 2.'''
    def __init__(self, embed_dim, out_dim=4, hidden=32, pairs=LEFT_RIGHT_PAIRS):
        super().__init__()
        self.pairs = list(pairs)
        self.f = nn.Sequential(nn.Linear(embed_dim, hidden), nn.GELU(), nn.Linear(hidden, out_dim))

    def per_joint_feature(self, tokens):
        # tokens: [B, SEGMENTS, 33, D] -> per-joint time-mean [B, 33, D]
        return tokens.mean(dim=1)

    def s_from_perjoint(self, pj):
        out = 0.0
        for li, ri in self.pairs:
            out = out + (self.f(pj[:, li, :]) - self.f(pj[:, ri, :]))
        return out

    def forward(self, tokens):
        return self.s_from_perjoint(self.per_joint_feature(tokens))

    def swapped(self, tokens):
        '''Wiring self-check ONLY (swap the head's own inputs). By construction returns -forward(tokens);
        it is NOT the training loss, because it trains nothing.'''
        pj = self.per_joint_feature(tokens)
        pj_sw = pj.clone()
        for li, ri in self.pairs:
            pj_sw[:, li, :] = pj[:, ri, :]
            pj_sw[:, ri, :] = pj[:, li, :]
        return self.s_from_perjoint(pj_sw)


def encoder_tokens_for_head(view_encoder, coords, segments, joints, embed_dim):
    '''Run raw coords through the VIEW encoder and reshape to [B, SEGMENTS, 33, D] for the head.'''
    return view_encoder(coords).reshape(len(coords), segments, joints, embed_dim)


def equivariance_loss(head, view_encoder, coords, segments, embed_dim, joints=33):
    '''Label-free equivariance term THROUGH the encoder:
        L_equiv = mean( ( s(encoder(Mx)) + s(encoder(x)) )^2 ),  M = anatomical mirror on raw coords.
    Zero only when the encoder represents the mirrored body as the sign-flip of the original along the
    head's axis. This is a real constraint on the ENCODER (the encoder is not equivariant by construction).'''
    tok = encoder_tokens_for_head(view_encoder, coords, segments, joints, embed_dim)
    tok_m = encoder_tokens_for_head(view_encoder, anatomical_mirror_coords(coords), segments, joints, embed_dim)
    s = head(tok)
    s_m = head(tok_m)
    return ((s_m + s) ** 2).mean()


# ---- guardrail (a): head negates under a swap of its OWN input, by construction (wiring self-check) ----
torch.manual_seed(RANDOM_SEED)
_head = AntisymmetricHead(32, out_dim=HEAD_OUT_DIM)
_tok = torch.randn(6, 8, 33, 32)
with torch.no_grad():
    _s, _ssw = _head(_tok), _head.swapped(_tok)
_slope = float(np.polyfit(_s.reshape(-1).numpy(), _ssw.reshape(-1).numpy(), 1)[0])
assert abs(_slope + 1.0) < 1e-4, f"head must negate under its own-input swap; slope={_slope}"
print(f"guardrail (a): head own-input swap slope = {_slope:+.6f}  (exact -1 by construction; NOT the loss)")

# ---- guardrail (b): the REAL loss is nonzero with nonzero grads into a deliberately NON-equivariant encoder ----
# A random Linear over the raw coords is not equivariant, so L_equiv through it must be > 0 with real grads.
torch.manual_seed(RANDOM_SEED + 3)
_fake_frames, _fake_seg, _fake_dim = 8, 2, 32
class _NonEquivEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(_fake_frames * 3, _fake_seg * _fake_dim)
    def forward(self, x):  # x: [B, F, 33, 3] -> [B, SEG*33, D]
        b = len(x)
        per_joint = x.permute(0, 2, 1, 3).reshape(b, 33, _fake_frames * 3)
        out = self.proj(per_joint).reshape(b, 33, _fake_seg, _fake_dim)
        return out.permute(0, 2, 1, 3).reshape(b, _fake_seg * 33, _fake_dim)
_enc = _NonEquivEncoder()
_head_b = AntisymmetricHead(_fake_dim, out_dim=HEAD_OUT_DIM)
_coords = torch.randn(4, _fake_frames, 33, 3)
_le = equivariance_loss(_head_b, _enc, _coords, _fake_seg, _fake_dim)
_le.backward()
_gh = sum(float(p.grad.norm()) for p in _head_b.parameters() if p.grad is not None)
_ge = sum(float(p.grad.norm()) for p in _enc.parameters() if p.grad is not None)
assert float(_le) > 1e-8, f"L_equiv must be > 0 on a non-equivariant encoder; got {float(_le):.3e} (no-op!)"
assert _gh > 1e-8 and _ge > 1e-8, f"L_equiv must push head AND encoder; head|grad|={_gh:.3e} enc|grad|={_ge:.3e}"
print(f"guardrail (b): L_equiv={float(_le):.4e} (> 0)  head|grad|={_gh:.3e}  encoder|grad|={_ge:.3e}  (real, not a no-op)")

guardrail (a): head own-input swap slope = -1.000000  (exact -1 by construction; NOT the loss)
guardrail (b): L_equiv=1.6171e-01 (> 0)  head|grad|=1.586e+00  encoder|grad|=8.008e-01  (real, not a no-op)


/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/3228400976.py:93: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:823.)
  assert float(_le) > 1e-8, f"L_equiv must be > 0 on a non-equivariant encoder; got {float(_le):.3e} (no-op!)"


## 3. Smoke cohort with a planted, sign-flipping lateral lean (reused from nb_09a)

**Step 4 of 9.**

*What we are about to do.* Build a small SYNTHETIC cohort: one hand-authored gait fixture per condition,
with a lateral lean planted into it whose SIGN alternates between the two synthetic sources of each
condition, and whose magnitude is largest for the folders labelled lateralized and exactly zero for
myopathic and normal.

*Why a planted signal is the right choice for a scaffold.* The purpose here is to exercise the machinery,
and machinery cannot be exercised on a cohort with no structure: the equivariance term and the readout
both need a genuine signed quantity to organise around, otherwise a passing run would prove only that the
code does not crash. Planting the signal means we KNOW the answer in advance, so a failure to recover it
is a plumbing bug rather than a finding.

*Why that same choice makes every downstream number non-evidential.* The signal is planted by us, at a
magnitude we chose, with a sign we assigned. Recovering it says nothing about real gait, and failing to
recover it says nothing about the real encoder. This is a fixture, not physiology.

*The important structural difference from the real cohort.* This cohort has 2 synthetic sources per
condition, 10 in total, with 3 clips each. The real cohort in `nb_09a` has 18 real source videos with
heavily unbalanced per-condition counts. So even the fold structure here is not comparable to the real
one.

*What to look at in the output.* Five lines showing `xyz (6, 32, 33, 3)` and `sources 2` per condition:
6 clips of 32 frames each, over 2 sources. Note the frame count is 32, not the real lineage's 64, which is
another reason the numbers below are not comparable to a real run.


In [4]:
def synthetic_gait_sequence(condition="normal", frames=64, seed=0):
    rng = np.random.default_rng(seed)
    phase = np.linspace(0.0, 4.0 * np.pi, frames, endpoint=False)
    seq = np.zeros((frames, 33, 4), dtype=np.float32)
    seq[..., 3] = 1.0
    base = {11: (0.42, 0.28), 12: (0.58, 0.28), 23: (0.45, 0.52), 24: (0.55, 0.52),
            25: (0.44, 0.70), 26: (0.56, 0.70), 27: (0.43, 0.89), 28: (0.57, 0.89),
            29: (0.42, 0.92), 30: (0.58, 0.92), 31: (0.39, 0.94), 32: (0.61, 0.94)}
    for joint, (x, y_) in base.items():
        seq[:, joint, 0] = x
        seq[:, joint, 1] = y_
    amplitude, lift = 0.045, 0.025
    if condition == "parkinsons":
        amplitude *= 0.45; lift *= 0.45
    if condition == "myopathic":
        seq[:, [11, 12], 0] += 0.03 * np.sin(phase)[:, None]
        seq[:, [23, 24], 0] += 0.018 * np.sin(phase)[:, None]
    for joint, knee, foot, offset in [(27, 25, 31, 0.0), (28, 26, 32, np.pi)]:
        wave = np.sin(phase + offset)
        if condition == "stroke" and joint == 27:
            wave = 0.35 * wave
        if condition == "cerebralpalsy":
            seq[:, knee, 1] -= 0.045; seq[:, joint, 1] -= 0.02
        seq[:, joint, 0] += amplitude * wave
        seq[:, knee, 0] += 0.4 * amplitude * wave
        seq[:, foot, 0] += amplitude * wave
        seq[:, joint, 1] -= lift * np.maximum(wave, 0.0)
        seq[:, foot, 1] -= 0.7 * lift * np.maximum(wave, 0.0)
    seq[..., :3] += rng.normal(0.0, 0.0025, seq[..., :3].shape)
    return seq


SMOKE_LEAN_MAGNITUDE = {"stroke": 1.0, "cerebralpalsy": 0.8, "parkinsons": 0.5, "myopathic": 0.0, "normal": 0.0}


def plant_signed_lean(seq, sign, magnitude, seed):
    if magnitude == 0.0 or sign == 0:
        return seq
    rng = np.random.default_rng(seed)
    out = seq.copy()
    phase = np.linspace(0.0, 4.0 * np.pi, len(seq), endpoint=False)
    for li, ri in LEFT_RIGHT_PAIRS:
        gain = 0.03 * magnitude * sign * rng.uniform(0.8, 1.2)
        out[:, li, 0] += gain * np.sin(phase)
        out[:, ri, 0] -= gain * np.sin(phase)
    return out


def center_and_scale_min(seq):
    xyz = np.asarray(seq, dtype=np.float32)[..., :3].copy()
    pelvis = 0.5 * (xyz[:, 23] + xyz[:, 24])
    xyz = xyz - pelvis[:, None, :]
    sh = np.linalg.norm(xyz[:, 11, :2] - xyz[:, 12, :2], axis=-1)
    hp = np.linalg.norm(xyz[:, 23, :2] - xyz[:, 24, :2], axis=-1)
    scale = np.nanmedian(np.maximum(sh, hp))
    if not np.isfinite(scale) or scale < 1e-6:
        scale = 1.0
    return np.nan_to_num(xyz / scale).astype(np.float32)


FRAMES = int(os.getenv("SJEPA_FRAMES", "32"))
SEGMENT_LENGTH = 4
SEGMENTS = FRAMES // SEGMENT_LENGTH


def build_smoke_condition_data(clips_per_source=3, frames=FRAMES):
    data, counter = {}, 0
    for condition in CONDITIONS:
        xyz_list, valid_list, vids = [], [], []
        for source in range(2):
            sign = 1 if source == 0 else -1
            mag = SMOKE_LEAN_MAGNITUDE[condition]
            for clip in range(clips_per_source):
                base = synthetic_gait_sequence(condition=condition, frames=frames, seed=RANDOM_SEED + counter)
                seq = plant_signed_lean(base, sign, mag, seed=RANDOM_SEED + 1000 + counter)
                xyz = center_and_scale_min(seq)
                xyz_list.append(xyz)
                valid_list.append(np.ones((frames, 33), dtype=bool))
                vids.append(f"smoke_source_{condition}_{source}")
                counter += 1
        data[condition] = {"xyz": np.stack(xyz_list).astype(np.float32),
                           "valid": np.stack(valid_list),
                           "video_ids": vids,
                           "records": [{"sequence_id": f"{condition}_{i}", "video_id": v}
                                       for i, v in enumerate(vids)]}
    return data


condition_data = build_smoke_condition_data()
for c in CONDITIONS:
    print(f"{c:14s}: xyz {condition_data[c]['xyz'].shape}  sources {len(set(condition_data[c]['video_ids']))}")

normal        : xyz (6, 32, 33, 3)  sources 2
parkinsons    : xyz (6, 32, 33, 3)  sources 2
stroke        : xyz (6, 32, 33, 3)  sources 2
myopathic     : xyz (6, 32, 33, 3)  sources 2
cerebralpalsy : xyz (6, 32, 33, 3)  sources 2


## 4. Training helpers (adapted from notebook 04's stage loop)

**Step 5 of 9.**

*What we are about to do.* Define the pooling, regularisation, batching, and masking helpers the stage
loop needs: `authorized_pool`, `vicreg_terms`, `condition_group_terms`, `balanced_epoch_batches`, and
`uniform_authorized_mask`.

*Why they are adapted rather than reused verbatim.* The training loop needs a self-contained masking path
so the smoke ladder runs without the real manifest, so `uniform_authorized_mask` is a simplified uniform
mask over the 12 authorized joints rather than notebook 04's full scheme. That is a deliberate
simplification of the scaffold, and it is one more reason this notebook's numbers are not comparable to a
real run.

*The one substantive change from notebook 04, stated precisely.* After the JEPA, VICReg, and group losses
are computed, when `equiv_on` is True we additionally compute `L_equiv` by running the raw batch AND its
anatomical mirror through the VIEW encoder, then add `EQUIV_WEIGHT * L_equiv` to the total loss. Nothing
else about the loss changes.

*Two invariants the code enforces, and why each matters.* The antisymmetric head's parameters join the
AdamW trainable set, because a head that cannot move could not express the equivariance the term is asking
for. And the TARGET encoder stays frozen, asserted at every single step, because the target encoder is
updated only by exponential moving average from the view encoder; if a gradient ever reached it, the JEPA
objective would collapse into predicting its own moving target.

*Terminology reminder, since two log fields are easy to misread.* The abbreviated `group` field in
notebook 04's logs is only the centroid-margin penalty, not the complete group loss. The abbreviated `std`
field is the mean feature standard deviation of unprojected EMA-teacher embeddings, and it is a
DIAGNOSTIC, not a VICReg component and not a loss.

*What to look at in the output.* The single line `training helpers ready.`


In [5]:
def authorized_pool(tokens, valid_patch):
    batch, segments, _, dim = tokens.shape
    selected = tokens[:, :, MASK_KEYPOINTS].reshape(batch, -1, dim)
    weights = valid_patch[:, :, MASK_KEYPOINTS].reshape(batch, -1).to(tokens.dtype)
    denom = weights.sum(dim=1, keepdim=True).clamp_min(1.0)
    return (selected * weights.unsqueeze(-1)).sum(dim=1) / denom


def off_diagonal(m):
    n, _ = m.shape
    return m.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()


def vicreg_terms(a, b, gamma=1.0, eps=1e-4):
    invariance = F.mse_loss(a, b)
    a_std = torch.sqrt(a.var(dim=0, unbiased=False) + eps)
    b_std = torch.sqrt(b.var(dim=0, unbiased=False) + eps)
    variance = 0.5 * (F.relu(gamma - a_std).mean() + F.relu(gamma - b_std).mean())
    ac, bc = a - a.mean(dim=0), b - b.mean(dim=0)
    denom = max(len(a) - 1, 1)
    cov = (off_diagonal(ac.T @ ac / denom).square().sum()
           + off_diagonal(bc.T @ bc / denom).square().sum()) / (2.0 * a.shape[1])
    return 25.0 * invariance + 25.0 * variance + cov


def condition_group_terms(reps, cond_ids, margin=1.0):
    unique = torch.unique(cond_ids)
    zero = reps.sum() * 0.0
    if len(unique) < 2:
        return zero, zero
    norm = F.normalize(reps, dim=1)
    centroids = torch.stack([F.normalize(norm[cond_ids == v].mean(dim=0), dim=0) for v in unique])
    compact = torch.stack([(norm[cond_ids == v] - centroids[i]).square().sum(dim=1).mean()
                           for i, v in enumerate(unique)]).mean()
    pw = (centroids[:, None] - centroids[None, :]).square().sum(dim=-1).clamp_min(1e-12).sqrt()
    upper = torch.triu(torch.ones_like(pw, dtype=torch.bool), diagonal=1)
    sep = F.relu(margin - pw[upper]).square().mean()
    return compact, sep


def balanced_epoch_batches(data, active, per_condition, rng):
    lengths = {c: len(data[c]["xyz"]) for c in active}
    steps = max(1, int(np.ceil(max(lengths.values()) / per_condition)))
    required = steps * per_condition
    orders = {}
    for c in active:
        pieces = []
        while sum(len(p) for p in pieces) < required:
            pieces.append(rng.permutation(lengths[c]))
        orders[c] = np.concatenate(pieces)[:required]
    for step in range(steps):
        xs, vs, ls = [], [], []
        for label, c in enumerate(active):
            take = orders[c][step * per_condition:(step + 1) * per_condition]
            xs.append(data[c]["xyz"][take]); vs.append(data[c]["valid"][take]); ls.extend([label] * per_condition)
        perm = rng.permutation(len(ls))
        yield (np.concatenate(xs)[perm], np.concatenate(vs)[perm], np.asarray(ls, dtype=np.int64)[perm])


def uniform_authorized_mask(valid_patch, frac, seed):
    eligible_joint = np.zeros(33, dtype=bool); eligible_joint[MASK_KEYPOINTS] = True
    eligible = valid_patch & eligible_joint[None, None, :]
    counts = eligible.reshape(len(eligible), -1).sum(axis=1)
    n_mask = max(1, min(int(np.floor(counts.min() * frac)), int(counts.min()) - 1))
    rng = np.random.default_rng(seed)
    mask = np.zeros_like(eligible)
    for i in range(len(mask)):
        cand = np.flatnonzero(eligible[i].reshape(-1))
        mask[i].reshape(-1)[rng.choice(cand, size=n_mask, replace=False)] = True
    return mask


print("training helpers ready.")

training helpers ready.


## 5. One full curriculum run (returns a distinct-fingerprint checkpoint)

**Step 6 of 9.**

*What we are about to do.* Define `run_curriculum(seed, equiv_on)`, which builds a fresh tiny `SJEPAGait`
plus head, runs the five-stage curriculum, adds `EQUIV_WEIGHT * L_equiv` when `equiv_on` is True, monitors
collapse, and returns the model plus a checkpoint. Then run it once as a sanity check.

*Why the fingerprint payload includes the equivariance settings.* Because `equiv_weight` and `equiv_on`
are hashed into `dataset_fingerprint`, every rung gets its OWN fingerprint and can never be confused with
the baseline lineage or with the other rung. Without that, two checkpoints trained under different
objectives could carry the same identifier, and a downstream evaluation could silently score the wrong
one.

*Why the labels `DATA_SOURCE` and `TRAINING_SCALE` exist, and the confusion they were added to prevent.*
`MODE` is only the artifact directory, taken from `GAVD_MODE`. It says nothing about the data or the
training scale. An earlier version of this bundle was labelled by `MODE` alone, which meant it could read
`real` while carrying two-epoch numbers from planted toy sequences. So the code now sets
`DATA_SOURCE = "synthetic"`, derives `TRAINING_SCALE` by comparing the config against the real scale
(embed_dim 96, encoder_depth 4, predictor_depth 2, and at least 300 stage-0 epochs), and then ASSERTS
`TRAINING_SCALE == "smoke"`. That assertion is a guard against this notebook ever being mistaken for the
real run: a real-scale run belongs in `new_nb_09_02`, which checkpoints per rung.

*What the collapse monitor measures, defined before it is read.* `feature_std` is the mean per-dimension
standard deviation of pooled target-encoder features across the cohort, where HIGHER is safer because a
collapsed encoder maps everything to the same point. `mean_pair_cosine` is the average cosine similarity
between distinct pooled features, where LOWER is safer for the same reason.

*What to look at in the output, and what it tells you about this scaffold's health.*

- `artifact mode: real | data: synthetic | training scale: smoke`, with the parenthetical reminder that
  the bundle is labelled by the last two.
- `sanity run OK. fingerprint=355a5b784095`, its own lineage.
- The per-stage table. Read the two monitor columns closely: `feature_std` runs from **0.002857** at stage
  0 to **0.010836** at stage 4, and `mean_pair_cosine` stays between **0.999548** and **0.999966**. A mean
  pairwise cosine of 0.9995 means essentially every sequence is mapped to nearly the same direction. This
  tiny model, at this scale, is very close to COLLAPSED. That is expected at 2 epochs on 6 clips per
  condition, and it is a third independent reason the R-squared values in step 8 are plumbing checks
  rather than measurements: there is barely any representation there to read.
- `l_equiv_last` hovers between 0.000654 and 0.008246 with no clear trend, which is consistent with the
  term having little purchase at this scale.
- Two benign `enable_nested_tensor` warnings from PyTorch.


In [6]:
SMOKE_EPOCHS_STAGE0 = int(os.getenv("IDEA9_SMOKE_EPOCHS0", "2"))
SMOKE_EPOCHS_FT = int(os.getenv("IDEA9_SMOKE_EPOCHS_FT", "1"))
SAMPLES_PER_CONDITION = int(os.getenv("IDEA9_SAMPLES_PER_CONDITION", "2"))
SMOKE_CONFIG = {"frames": FRAMES, "joints": 33, "coordinate_dim": 3, "segment_length": SEGMENT_LENGTH,
                "embed_dim": 32, "encoder_depth": 1, "predictor_depth": 1, "heads": 4}
EMBED_DIM = SMOKE_CONFIG["embed_dim"]

# Two labels that are easy to confuse, and were confused in an earlier version of this bundle.
# MODE is only the artifact directory this notebook writes into, taken from GAVD_MODE. It says
# nothing about either the data or the training scale here: section 3 always builds a SYNTHETIC
# cohort, and this notebook always trains at smoke scale. A bundle labelled by MODE alone can
# therefore read "real" while carrying two-epoch numbers from planted toy sequences.
DATA_SOURCE = "synthetic"
REAL_SCALE = {"embed_dim": 96, "encoder_depth": 4, "predictor_depth": 2}
TRAINING_SCALE = ("real" if all(SMOKE_CONFIG[k] == v for k, v in REAL_SCALE.items())
                  and SMOKE_EPOCHS_STAGE0 >= 300 else "smoke")
assert TRAINING_SCALE == "smoke", (
    "nb_09b is the smoke-scale plumbing notebook. A real-scale run belongs in "
    "new_nb_09_02_real_multiseed_equivariant_training.ipynb, which checkpoints per rung.")
print(f"artifact mode: {MODE} | data: {DATA_SOURCE} | training scale: {TRAINING_SCALE} "
      "(the bundle is labelled by the last two, not the first)")
device = torch.device("cpu")


def collapse_monitor(model, data, active):
    arrays = np.concatenate([data[c]["xyz"] for c in active])
    valid = np.concatenate([data[c]["valid"] for c in active])
    model.target_encoder.eval()
    with torch.no_grad():
        x = torch.tensor(arrays, dtype=torch.float32, device=device)
        vp = torch.tensor(valid, dtype=torch.bool, device=device).reshape(
            len(x), SEGMENTS, SEGMENT_LENGTH, 33).all(dim=2)
        tok = model.target_encoder(x).reshape(len(x), SEGMENTS, 33, EMBED_DIM)
        pooled = authorized_pool(tok, vp)
        std = float(pooled.std(dim=0, unbiased=False).mean())
        unit = F.normalize(pooled, dim=1)
        cos = unit @ unit.T
        eye = torch.eye(len(unit), dtype=torch.bool)
        mpc = float(cos[~eye].mean()) if len(unit) > 1 else float("nan")
    return {"feature_std": std, "mean_pair_cosine": mpc}


def run_curriculum(seed, equiv_on):
    torch.manual_seed(seed); np.random.seed(seed)
    model = SJEPAGait(**SMOKE_CONFIG).to(device)
    head = AntisymmetricHead(EMBED_DIM, out_dim=HEAD_OUT_DIM).to(device)

    class VProj(nn.Module):
        def __init__(self, d):
            super().__init__(); self.net = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, d))
        def forward(self, x): return self.net(x)
    vproj = VProj(EMBED_DIM).to(device)

    assert not any(p.requires_grad for p in model.target_encoder.parameters())
    history, global_step_all = [], 0
    l_equiv_last = float("nan")
    for stage in CURRICULUM:
        active = stage["conditions"]
        epochs = SMOKE_EPOCHS_STAGE0 if stage["stage"] == 0 else SMOKE_EPOCHS_FT
        trainable = [*[p for p in model.view_encoder.parameters() if p.requires_grad],
                     *[p for p in model.predictor.parameters() if p.requires_grad],
                     *list(vproj.parameters()), *list(head.parameters())]
        opt = torch.optim.AdamW(trainable, lr=3e-4, betas=(0.9, 0.95), weight_decay=0.05)
        steps_per_epoch = max(1, int(np.ceil(max(len(condition_data[c]["xyz"]) for c in active) / SAMPLES_PER_CONDITION)))
        total_steps = epochs * steps_per_epoch
        rng = np.random.default_rng(seed + 1000 * stage["stage"])
        for epoch in range(epochs):
            model.train(); vproj.train(); head.train()
            for xyz_np, valid_np, labels_np in balanced_epoch_batches(condition_data, active, SAMPLES_PER_CONDITION, rng):
                coords = torch.tensor(xyz_np, dtype=torch.float32, device=device)
                valid = torch.tensor(valid_np, dtype=torch.bool, device=device)
                labels = torch.tensor(labels_np, dtype=torch.long, device=device)
                vp = valid.reshape(len(valid), SEGMENTS, SEGMENT_LENGTH, 33).all(dim=2)
                mask_np = uniform_authorized_mask(vp.cpu().numpy(), MASK_FRACTION, seed=seed + 100000 * stage["stage"] + global_step_all)
                target_mask = torch.tensor(mask_np, dtype=torch.bool, device=device)
                view_a = geometric_view(coords, flip_probability=0.0)
                view_b = geometric_view(coords, flip_probability=0.0)
                prediction, target = model(view_a, coords, target_mask)
                jepa_loss = sjepa_cross_entropy(prediction, target, model.target_center)
                tok_a = model.view_encoder(view_a).reshape(len(view_a), SEGMENTS, 33, EMBED_DIM)
                tok_b = model.view_encoder(view_b).reshape(len(view_b), SEGMENTS, 33, EMBED_DIM)
                vic = vicreg_terms(vproj(authorized_pool(tok_a, vp)), vproj(authorized_pool(tok_b, vp)))
                compact, sep = condition_group_terms(authorized_pool(tok_a, vp), labels, margin=GROUP_MARGIN)
                total_loss = jepa_loss + VICREG_WEIGHT * vic + GROUP_WEIGHT * (compact + sep)
                if equiv_on:
                    # L_equiv runs the RAW skeleton and its anatomical mirror through the VIEW encoder,
                    # then penalizes s(enc(Mx)) + s(enc(x)). This shapes the ENCODER (the head alone is
                    # already antisymmetric, so a head-only swap would be an identically-zero no-op).
                    l_equiv = equivariance_loss(head, model.view_encoder, coords, SEGMENTS, EMBED_DIM)
                    total_loss = total_loss + EQUIV_WEIGHT * l_equiv
                    l_equiv_last = float(l_equiv.detach())
                if not torch.isfinite(total_loss):
                    raise FloatingPointError(f"non-finite loss seed={seed} stage={stage['stage']} step={global_step_all}")
                opt.zero_grad(set_to_none=True)
                total_loss.backward()
                assert all(p.grad is None for p in model.target_encoder.parameters())
                torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
                opt.step()
                model.update_target(cosine_ema(global_step_all, max(total_steps, 1), start=0.996, end=1.0))
                model.update_center(target, beta=0.9)
                global_step_all += 1
        mon = collapse_monitor(model, condition_data, active)
        history.append({"seed": seed, "equiv_on": equiv_on, "stage": stage["stage"],
                        "feature_std": mon["feature_std"], "mean_pair_cosine": mon["mean_pair_cosine"],
                        "l_equiv_last": l_equiv_last})
    model.eval()
    payload = {"mode": MODE, "seed": seed, "equiv_on": equiv_on, "equiv_weight": EQUIV_WEIGHT,
               "config": SMOKE_CONFIG, "curriculum": [s["name"] for s in CURRICULUM],
               "vicreg_weight": VICREG_WEIGHT, "group_weight": GROUP_WEIGHT}
    fingerprint = hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()
    checkpoint = {"model_state": model.state_dict(), "head_state": head.state_dict(),
                  "config": SMOKE_CONFIG, "mode": MODE, "mask_keypoints": MASK_KEYPOINTS,
                  "curriculum_complete": True, "conditions_seen": CONDITIONS,
                  "dataset_fingerprint": fingerprint, "fingerprint_payload": payload,
                  "equiv_on": equiv_on, "equiv_weight": EQUIV_WEIGHT if equiv_on else 0.0}
    return model, head, checkpoint, history


# Unconditional: the run is seconds at this scale, and gating it on GAVD_MODE used to leave the
# notebook with two silent sections whenever it was pointed at the real artifact directory.
_m, _h, _ck, _hist = run_curriculum(seed=SEEDS[0], equiv_on=True)
print(f"sanity run OK. fingerprint={_ck['dataset_fingerprint'][:12]} (its own lineage, never the baseline's)")
print(pd.DataFrame(_hist)[["stage", "feature_std", "mean_pair_cosine", "l_equiv_last"]].to_string(index=False))

artifact mode: real | data: synthetic | training scale: smoke (the bundle is labelled by the last two, not the first)


/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:65: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)


sanity run OK. fingerprint=355a5b784095 (its own lineage, never the baseline's)
 stage  feature_std  mean_pair_cosine  l_equiv_last
     0     0.002857          0.999966      0.008246
     1     0.003823          0.999942      0.000654
     2     0.004225          0.999938      0.004012
     3     0.004031          0.999945      0.002668
     4     0.010836          0.999548      0.003053


## 6. Re-score each checkpoint through the arm 1 instrument

**Step 7 of 9.**

*What we are about to do.* Score every retrained checkpoint with the SAME antisymmetric-head readout and
the SAME source-disjoint ridge probe that `nb_09a` uses, and also record each retrained encoder's measured
anatomical-mirror slope.

*Why the instrument must be shared.* D0 and E1 differ by one loss term, so they must be judged on one
ruler; otherwise any difference could come from the evaluation rather than from the treatment. Reusing
`nb_09a`'s instrument also means the ladder speaks in the same units as arm 1's ladder.

*What each piece is.* The target `y` is the raw-coordinate `signed_left_minus_right`. The feature is the
antisymmetric head `s` on the retrained TARGET encoder's tokens. The probe is `GroupKFold` on `video_id`
with an inner alpha choice on training sources only, exactly as in arm 1.

*The mirror slope reported here is MEASURED, not the exact -1.* It runs through the encoder, so it is a
property of the encoder. The exact -1 belongs to the head's own input swap and is a separate quantity, as
section 2 explained.

*What to look at in the output.* `re-score of the sanity run: R2=-0.379 measured mirror slope=+0.449`.
Note the mirror slope is POSITIVE here. A mirror-equivariant encoder would give roughly -1, and 0 would
mean no relationship, so +0.449 means this smoke encoder responds to a mirrored body somewhat like the
original rather than like its negation. On a near-collapsed toy encoder that is unsurprising and carries
no information about the real encoder.

*What we may NOT conclude.* Nothing about any checkpoint of interest. This is one synthetic run of a tiny
model, reported to show the instrument returns numbers.


In [7]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
ALPHAS = np.logspace(-3, 3, 13)


def signed_left_minus_right(coords):
    coords = np.asarray(coords, dtype=np.float64)[..., :3]
    total = 0.0
    for li, ri in LEFT_RIGHT_PAIRS:
        total += coords[:, li, :].std(axis=0).sum() - coords[:, ri, :].std(axis=0).sum()
    return float(total)


def anatomical_mirror_xyz(coords):
    m = np.asarray(coords, dtype=np.float32).copy()
    m[:, :, 0] = -m[:, :, 0]
    for li, ri in FULL_LR_PAIRS:
        m[:, [li, ri], :] = m[:, [ri, li], :]
    return m


def eval_pool():
    xyz_list, y_list, groups = [], [], []
    for c in CONDITIONS:
        for i, xyz in enumerate(condition_data[c]["xyz"]):
            xyz_list.append(xyz); y_list.append(signed_left_minus_right(xyz))
            groups.append(condition_data[c]["video_ids"][i])
    return np.stack(xyz_list).astype(np.float32), np.array(y_list, dtype=np.float64), np.array(groups)


EVAL_XYZ, EVAL_Y, EVAL_GROUPS = eval_pool()


def antisym_features(model, head, xyz):
    model.target_encoder.eval(); head.eval()
    feats = []
    with torch.no_grad():
        for i in range(0, len(xyz), 16):
            x = torch.tensor(xyz[i:i + 16], dtype=torch.float32, device=device)
            tok = model.target_encoder(x).reshape(len(x), SEGMENTS, 33, EMBED_DIM)
            feats.append(head(tok).cpu().numpy())
    return np.concatenate(feats)


def source_disjoint_r2(X, y, groups):
    n_splits = max(2, min(5, len(np.unique(groups))))
    gkf = GroupKFold(n_splits=n_splits)
    preds = np.full(len(y), np.nan)
    for tr, te in gkf.split(X, y, groups):
        ig = groups[tr]; best_a = ALPHAS[0]; best = -np.inf
        if len(np.unique(ig)) >= 2:
            inner = GroupKFold(n_splits=min(3, len(np.unique(ig))))
            for a in ALPHAS:
                sc = []
                for itr, iva in inner.split(X[tr], y[tr], ig):
                    scaler = StandardScaler().fit(X[tr][itr])
                    mdl = Ridge(alpha=a).fit(scaler.transform(X[tr][itr]), y[tr][itr])
                    sc.append(r2_score(y[tr][iva], mdl.predict(scaler.transform(X[tr][iva]))))
                if np.mean(sc) > best:
                    best, best_a = np.mean(sc), a
        scaler = StandardScaler().fit(X[tr])
        mdl = Ridge(alpha=best_a).fit(scaler.transform(X[tr]), y[tr])
        preds[te] = mdl.predict(scaler.transform(X[te]))
    ok = ~np.isnan(preds)
    return float(r2_score(y[ok], preds[ok]))


def measured_mirror_slope(model, head, xyz):
    Xo = antisym_features(model, head, xyz)
    Xm = antisym_features(model, head, np.stack([anatomical_mirror_xyz(x) for x in xyz]))
    sc = StandardScaler().fit(Xo)
    probe = Ridge(alpha=1.0).fit(sc.transform(Xo), EVAL_Y)
    do, dm = probe.predict(sc.transform(Xo)), probe.predict(sc.transform(Xm))
    return float(np.polyfit(do, dm, 1)[0])


def score_checkpoint(model, head):
    X = antisym_features(model, head, EVAL_XYZ)
    return {"r2": source_disjoint_r2(X, EVAL_Y, EVAL_GROUPS),
            "mirror_slope": measured_mirror_slope(model, head, EVAL_XYZ)}


_sc = score_checkpoint(_m, _h)
print(f"re-score of the sanity run: R2={_sc['r2']:+.3f}  measured mirror slope={_sc['mirror_slope']:+.3f}")

re-score of the sanity run: R2=-0.379  measured mirror slope=+0.449


## 7. Run the ablation ladder D0 vs E1 across seeds and decide

**Step 8 of 9. The numbers this cell prints are PLUMBING VALIDATION ONLY, and the following paragraphs
say so before quoting any of them.**

*What we are about to do.* For each seed, run D0 (`equiv_on=False`) and E1 (`equiv_on=True`), score both
through step 7, then compare `mean(E1) - mean(D0)` against D0's seed-to-seed standard deviation.

*The credit rule, stated before the result.* The equivariance term earns credit only if the effect EXCEEDS
D0's seed-to-seed spread (the trajectory control, which asks whether the effect is bigger than the noise
of simply rerunning the recipe) AND clears the same 0.05 margin the primary gate uses. Requiring both
prevents crediting an effect that is merely larger than zero.

*What to look at in the output, with the correct reading attached.*

| Quantity | Value | What it means here |
|---|---|---|
| D0 mean R-squared | **-0.320** | control rung, synthetic and smoke scale |
| D0 seed spread | **0.036** | the trajectory-control yardstick |
| E1 mean R-squared | **-0.294** | treatment rung, synthetic and smoke scale |
| effect, E1 minus D0 | **+0.025** | smaller than the spread AND smaller than 0.05 |
| exceeds D0 spread | **False** | |
| beats floor 0.05 | **False** | |
| `L_equiv` earns credit | **False** | |

Also in the per-seed table: every rung has its own fingerprint, as designed, and the six mirror slopes
scatter wildly, from **-0.071** to **+1.301**. A quantity that should be near -1 for an equivariant
encoder is instead landing on both sides of zero across seeds, which is what an uninformative measurement
looks like.

*One detail in that table worth noticing.* `final_feature_std` and `final_mean_pair_cosine` are IDENTICAL
between D0 and E1 at every seed, to six decimals (0.010836 and 0.999548 at seed 0, 0.010879 and 0.999619
at seed 1, 0.011467 and 0.999648 at seed 2). The equivariance term left no trace at all in the collapse
monitors. That is consistent with the scale-degeneracy defect diagnosed in section 2 and detailed in
section 8: a term that can satisfy itself by shrinking the trainable head does not have to move the
encoder, and here the encoder's monitored statistics did not move.

*What we may conclude, stated plainly as a negative.* The ladder machinery works end to end: rungs run,
fingerprints separate, scores come back, and the credit rule arithmetic executes and returns False. That
is a plumbing success and nothing more.

*What we may NOT conclude, and this is the important half.* Nothing about the real encoder, in either
direction. This is a synthetic cohort with a planted signal, a tiny near-collapsed model, 2 stage-0
epochs, 32 frames, and a defective loss form. The uncredited effect of +0.025 is not evidence that
equivariance training does not work; on real data with a repaired term the endpoint moved by a factor of
about eight (see section 8). Equally, it is not evidence that it does work. The correct reading of this
cell is: the ladder is wired correctly, go read `new_nb_09_02` and `new_nb_09_03` for the result.

*Note that the code prints its warning unconditionally.* The
`SYNTHETIC DATA AT SMOKE SCALE ... NOT evidence about the real encoder` line used to be gated on
`GAVD_MODE`, which meant it stayed hidden in exactly the configuration where a reader is most likely to
mistake these numbers for real ones. It now always prints.


In [8]:
FLOOR_MARGIN = 0.05
ladder_rows = []
all_history = []
for seed in SEEDS:
    for equiv_on in (False, True):
        model, head, ck, hist = run_curriculum(seed=seed, equiv_on=equiv_on)
        sc = score_checkpoint(model, head)
        all_history.extend(hist)
        ladder_rows.append({"rung": "E1" if equiv_on else "D0", "seed": seed,
                            "r2": sc["r2"], "mirror_slope": sc["mirror_slope"],
                            "fingerprint": ck["dataset_fingerprint"][:12],
                            "final_feature_std": hist[-1]["feature_std"],
                            "final_mean_pair_cosine": hist[-1]["mean_pair_cosine"]})
        print(f"seed {seed} {'E1' if equiv_on else 'D0'}: R2={sc['r2']:+.3f} "
              f"mirror={sc['mirror_slope']:+.3f} fp={ck['dataset_fingerprint'][:8]}")

ladder = pd.DataFrame(ladder_rows)
d0_r2 = ladder[ladder["rung"] == "D0"]["r2"]
e1_r2 = ladder[ladder["rung"] == "E1"]["r2"]
d0_mean, d0_std = float(d0_r2.mean()), float(d0_r2.std(ddof=0))
e1_mean = float(e1_r2.mean())
effect = e1_mean - d0_mean
exceeds_spread = bool(effect > max(d0_std, 1e-9))
beats_floor = bool((e1_mean - d0_mean) >= FLOOR_MARGIN)
equiv_credited = bool(exceeds_spread and beats_floor)
print("\n" + ladder.to_string(index=False))
print(f"\nD0 mean R2 = {d0_mean:+.3f} (std {d0_std:.3f})   E1 mean R2 = {e1_mean:+.3f}")
print(f"effect E1-D0 = {effect:+.3f}   exceeds D0 spread: {exceeds_spread}   beats floor 0.05: {beats_floor}")
print(f"L_equiv earns credit: {equiv_credited}")
# Printed unconditionally. This warning used to be gated on GAVD_MODE, so it stayed hidden in exactly
# the configuration where a reader is most likely to mistake these numbers for real ones.
print(f"\n{DATA_SOURCE.upper()} DATA AT {TRAINING_SCALE.upper()} SCALE: this is a plumbing demonstration of "
      "the ladder, NOT evidence about the real encoder. The real ladder is new_nb_09_02.")

/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:65: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)


seed 0 D0: R2=-0.361 mirror=+0.456 fp=73c97b35


/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:65: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)


seed 0 E1: R2=-0.379 mirror=+0.449 fp=355a5b78


/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:65: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)


...

seed 1 E1: R2=-0.254 mirror=+1.301 fp=d43edbb2


/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:65: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)


seed 2 D0: R2=-0.273 mirror=-0.071 fp=0157d5f2


/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
/var/folders/7t/kps5880d0gz_kzh7t6rvsw280000gn/T/ipykernel_23056/920174020.py:65: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=depth)


seed 2 E1: R2=-0.250 mirror=-0.050 fp=547cbe93

rung  seed        r2  mirror_slope  fingerprint  final_feature_std  final_mean_pair_cosine
  D0     0 -0.360906      0.456415 73c97b35606e           0.010836                0.999548
  E1     0 -0.379101      0.449036 355a5b784095           0.010836                0.999548
  D0     1 -0.325501      1.019093 06ba926f9726           0.010879                0.999619
  E1     1 -0.254190      1.300827 d43edbb29a13           0.010879                0.999619
  D0     2 -0.272651     -0.071414 0157d5f2d4e0           0.011467                0.999648
  E1     2 -0.249871     -0.050421 547cbe937fb1           0.011467                0.999648

D0 mean R2 = -0.320 (std 0.036)   E1 mean R2 = -0.294
effect E1-D0 = +0.025   exceeds D0 spread: False   beats floor 0.05: False
L_equiv earns credit: False

SYNTHETIC DATA AT SMOKE SCALE: this is a plumbing demonstration of the ladder, NOT evidence about the real encoder. The real ladder is new_nb_09_02.


## 8. The real run happened elsewhere, and why this recipe was NOT followed

**Step 9 of 9.**

### 8a. SUPERSEDED: the recipe in the cell below must not be pasted into notebook 04 as written

*What the recipe was.* This section used to hold a plan for a deferred real run: paste section 2's head,
mirror, and equivariance loss into `04_pretrain_sjepa_on_normal.ipynb`, add the head to the trainable
list, run D0 and E1 rungs across seeds, and re-score each checkpoint through `nb_09a`. That run has since
been carried out, but **not** by following those steps.

*Why it must not be followed, stated as the defect rather than as a change of plan.* The head `s` in
section 2 is trainable and the loss is an ABSOLUTE squared residual, so the objective has a degenerate
solution: the head can shrink its own output toward zero and drive the term down **without the encoder
changing at all**.

*How badly, measured on synthetic fixtures in `new_nb_09_01`.* These are FIXTURE numbers from 30 synthetic
sequences over 30 epochs, not gait results:

- the equivariance term fell by a factor of **184.16**,
- while the head's output scale shrank by a factor of **4.77**,
- and the label-free mirror residual **rho** moved by only **0.0096**, against a gate of **0.0492**.

rho is scaled so that **0 is mirror equivariant and 4 is mirror blind**. So the term reported a
184-fold improvement in itself while the property it was supposed to install barely budged. Anyone
watching only the loss curve would have recorded a success and shipped it.

*The separate, even worse variant that must also not be used.* The head-only token-swap form of the loss,
penalising a swap of the head's OWN tokens, is an algebraic no-op with a loss of exactly **0.0** and a
gradient of exactly **0.0** into the head. It trains nothing whatsoever. Section 2 explains why.

*The repair.* Normalise the residual by the signal's own magnitude, so that shrinking the head scales
numerator and denominator together and buys nothing:

```text
L = mean_seq [ || s(enc(Mx)) + s(enc(x)) ||^2
             / ( 0.5 * ( || s(enc(x)) ||^2 + || s(enc(Mx)) ||^2 ) + eps ) ]
```

The real ladder went one step further and used the `parameter_free` variant, the same ratio with `s`
replaced by a FIXED antisymmetric contraction, so there are no head parameters left to shrink at all.

*Why the superseded recipe is KEPT in the cell below rather than deleted.* Deleting it would hide the
mistake, and the mistake is the most transferable thing in this notebook. It is kept verbatim, with a
`status` field reading `SUPERSEDED, DO NOT FOLLOW AS WRITTEN` attached to it, next to an `EXECUTED_PATH`
field naming what to do instead. Presenting the recipe WITHOUT that status field would invite someone to
repeat it.

### 8b. Where the real run lives

A separate four-notebook series carries out arm 2, leaving notebook 04 untouched so that the baseline
lineage and the experiment stay separable. Run them in this order:

1. `new_nb_09_00_methodology_and_contract` fixes the endpoints, guardrails, and credit rule in advance,
   and verifies the pose cache, mask whitelist, mapping hash, and baseline checkpoint. It trains nothing.
2. `new_nb_09_01_mechanism_and_smoke_validation` proves the term reaches the encoder, calibrates the
   endpoint against fixtures with known answers, and bakes off the loss variants. It is what exposed the
   degenerate solution described above, and it selects the variant the real run uses. All of its numbers
   are synthetic fixtures.
3. `new_nb_09_02_real_multiseed_equivariant_training` runs the real D0-versus-E1 ladder on the locked
   cohort, with the full five-stage curriculum per rung at 600 epochs and 11,400 optimizer updates.
4. `new_nb_09_03_evaluation_results_discussion` recomputes every endpoint and guardrail and applies the
   credit rule.

### 8c. What that run found: NO CREDIT

The cell below reads the verdict out of the series' own bundle rather than restating it, so this notebook
cannot drift from it. Its printed `real_run` block shows, and the arm 2 ladder ran **3 seeds against 5
registered**:

- primary endpoint rho on the target encoder: D0 mean **0.462**, E1 mean **0.0588**, improvement
  **0.403**, against a D0 seed spread of **0.0568**;
- credit rule: condition 1 (exceeds seed spread) **true**, condition 2 (paired bootstrap excludes zero)
  **true**, condition 3 (no guardrail regression) **false**, with `all_three_required` **true**;
- `failed_guardrails: ["feature_std"]`;
- `unmeasurable_guardrails: ["grouped_probe_balanced_accuracy"]`, that is the source-grouped five-class
  guardrail was **not evaluable** (never write that it failed);
- two recorded protocol deviations, on `ladder.seeds` and on the grouped guardrail;
- `verdict: "NO CREDIT"`.

*How to read that verdict, because it is a third distinct epistemic state and not a synonym for the other
two.* The effect is REAL, LARGE, and CONSISTENT: rho about eight times lower, roughly seven times the
control's seed spread, and improving on **18 of 18** source videos. And yet no credit is awarded, because
the guardrail failure is **not independent of the effect**. A term that asks the encoder to respond
identically to a body and its reflection is also a term that REMOVES VARIANCE. So variance loss is a live
competing explanation for the endpoint gain, and this experiment cannot separate "the encoder learned
mirror structure" from "the encoder lost variance in a way that happens to reduce rho". That is precisely
why the guardrail was registered in advance.

*The three verdicts, kept distinct.* Idea 5 returned an INFORMATIVE NULL: valid measurement, answer no.
Arm 1 returned an ARTIFACT: the lane is not admissible evidence about sides, so the claim is WITHDRAWN
rather than answered, which is a weaker epistemic state than a null. Arm 2 returns NO CREDIT: real effect,
failed guardrail, competing explanation. Never collapse them into "it did not work".

*One thing rho is not.* rho is a symmetry property of the representation. It is not accuracy, not class
separation, and not clinical value. A rho improvement is not evidence of downstream benefit, and arm 2's
own antisymmetric-lane R-squared did not improve.

### 8d. What this notebook remains good for

It is the plumbing scaffold. It demonstrates the ladder shape, the per-rung fingerprinting, the collapse
monitors, and the credit-rule arithmetic, on a synthetic cohort at smoke scale. Its ladder numbers are not
evidence about any encoder, and the bundle it writes is labelled by `data_source` and `training_scale` so
that they cannot be read as if they were.

*What to look at in the output.* The written bundle path, the `mode` / `data_source` / `artifact_mode` /
`seeds` / `decision` block (note `mode` reads `smoke` while `artifact_mode` reads `real`, which is the
whole point of the relabelling), and then the `real_run` block carrying the NO CREDIT verdict quoted
above.


In [9]:
# Kept verbatim as the plan of record, with the reason it must not be followed attached to it. Deleting it
# would hide the mistake; presenting it without the status field would invite someone to repeat it.
SUPERSEDED_RECIPE = {
    "status": "SUPERSEDED, DO NOT FOLLOW AS WRITTEN. Step 1 installs the absolute equivariance term below, "
              "which a trainable head can satisfy by shrinking its own output instead of changing the "
              "encoder. Use the scale-invariant term and the executable path in EXECUTED_PATH instead.",
    "step_1": "Paste AntisymmetricHead + anatomical_mirror_coords + equivariance_loss into nb04 training-step cell; add head params to trainable; add EQUIV_WEIGHT*equivariance_loss(head, model.view_encoder, coords, SEGMENTS, EMBED_DIM) to total_loss behind equiv_on. Loss runs raw coords AND their anatomical mirror THROUGH the encoder; a head-only token swap is a no-op.",
    "step_2": "Add equiv_weight and equiv_on to fingerprint_payload (each checkpoint gets its own fingerprint; never the baseline's).",
    "step_3_D0": "GAVD_MODE=real SJEPA_RUN_PROFILE=recommended IDEA9_EQUIV_WEIGHT=0 across seeds 0,1,2[,3,4].",
    "step_3_E1": "GAVD_MODE=real SJEPA_RUN_PROFILE=recommended IDEA9_EQUIV_WEIGHT=0.02 across the same seeds.",
    "step_4": "Re-score every checkpoint through nb_09a (SJEPA_INSPECT_CHECKPOINT=<file>); apply the section-7 credit rule (E1-D0 must exceed D0 seed spread AND clear the 0.05 floor).",
}
EXECUTED_PATH = {
    "notebooks_in_order": [
        "new_nb_09_00_methodology_and_contract",
        "new_nb_09_01_mechanism_and_smoke_validation",
        "new_nb_09_02_real_multiseed_equivariant_training",
        "new_nb_09_03_evaluation_results_discussion",
    ],
    "why_not_this_scaffold": "section 2's absolute term is satisfiable by shrinking the trainable head, so "
                            "the real run uses a scale-invariant form; notebook 04 was left untouched so "
                            "the baseline lineage and the experiment stay separable",
    "endpoint": "rho, a parameter-free normalized mirror residual read with an identity head, so no "
                "trainable weight can influence it (0 = mirror equivariant, 4 = mirror blind)",
}


def real_run_outcome():
    # Read the superseding series' verdict instead of restating it, so this bundle cannot drift from it.
    path = OUT_DIR / "idea9_arm2" / "idea9_arm2_evaluation_result.json"
    if not path.is_file():
        return {"status": "not present in this artifact root", "expected_bundle": str(path)}
    result = json.loads(path.read_text(encoding="utf-8"))
    primary = result["primary"]
    return {
        "status": "completed",
        "bundle": str(path.relative_to(OUT_DIR)),
        "verdict": result["PRIMARY_VERDICT"],
        "seeds_per_rung": len({row["seed"] for row in result["per_rung"]}),
        "rho_D0_mean": primary["D0_mean"],
        "rho_E1_mean": primary["E1_mean"],
        "improvement": primary["improvement"],
        "D0_seed_spread": primary["D0_seed_spread"],
        "credit_rule": result["credit_rule"],
        "failed_guardrails": [row["guardrail"] for row in result["guardrails"]
                              if row["measurable"] and not row["within_control_spread"]],
        "unmeasurable_guardrails": [row["guardrail"] for row in result["guardrails"] if not row["measurable"]],
        "protocol_deviations": [entry["field"] for entry in result.get("protocol_deviations", [])],
    }


REAL_RUN = real_run_outcome()
bundle = {
    "notebook": "nb_09b_equivariant_retrain",
    "arm": "arm2_equivariance_coupled_retrain",
    "role": "smoke-scale plumbing scaffold; superseded for results by the new_nb_09 series",
    "mode": TRAINING_SCALE,
    "data_source": DATA_SOURCE,
    "training_scale": TRAINING_SCALE,
    "artifact_mode": MODE,
    "equiv_weight": EQUIV_WEIGHT,
    "seeds": SEEDS,
    "total_loss": f"jepa + {VICREG_WEIGHT}*vicreg + {GROUP_WEIGHT}*group + {EQUIV_WEIGHT}*L_equiv",
    "L_equiv": "mean((s(encoder(Mx)) + s(encoder(x)))^2), M = anatomical mirror on raw coords, run THROUGH the view encoder, label-free (no L_axis). A head-only token swap would be identically zero (no-op). DEGENERATE: a trainable s can shrink its output to satisfy this without changing the encoder; the real run uses a scale-invariant form instead.",
    "ladder": ladder_rows,
    "collapse_history": all_history,
    "decision": {"d0_mean_r2": d0_mean, "d0_std_r2": d0_std, "e1_mean_r2": e1_mean,
                 "effect_E1_minus_D0": effect, "exceeds_D0_spread": exceeds_spread,
                 "beats_floor_0.05": beats_floor, "equiv_credited": equiv_credited},
    "superseded_recipe": SUPERSEDED_RECIPE,
    "executed_path": EXECUTED_PATH,
    "real_run": REAL_RUN,
    "deferred": REAL_RUN["status"] != "completed",
    "notes": "These numbers are plumbing checks on a SYNTHETIC cohort at SMOKE scale, not a result: "
             "`data_source` and `training_scale` say so, `artifact_mode` is only the directory written to, "
             "and `mode` repeats the training scale for readers of older bundles that used it for the "
             "cache. The real multi-seed result is `real_run` above, produced by new_nb_09_02 and "
             "adjudicated by new_nb_09_03 with a repaired scale-invariant term. The token-swap slope -1 is "
             "exact by construction; the anatomical-mirror slope is measured through the encoder. All "
             "results transductive; source video is the independent unit; folder labels are dataset "
             "annotations, not diagnoses.",
}
bundle_path = OUT_DIR / "idea9_equivariant_retrain_result.json"
bundle_path.write_text(json.dumps(bundle, indent=2))
print(f"wrote {bundle_path}")
print(json.dumps({k: bundle[k] for k in ("mode", "data_source", "artifact_mode", "seeds", "decision")}, indent=2))
print("\nthe real run this scaffold points at:")
print(json.dumps(REAL_RUN, indent=2))

wrote /Users/pmui/dev/alexpose/experiments/sjepa/gavd6/work/artifacts/real/idea9_equivariant_retrain_result.json
{
  "mode": "smoke",
  "data_source": "synthetic",
  "artifact_mode": "real",
  "seeds": [
    0,
    1,
    2
  ],
  "decision": {
    "d0_mean_r2": -0.31968620787329355,
    "d0_std_r2": 0.03626390165425014,
    "e1_mean_r2": -0.2943872236758502,
    "effect_E1_minus_D0": 0.025298984197443364,
    "exceeds_D0_spread": false,
    "beats_floor_0.05": false,
    "equiv_credited": false
  }
}

the real run this scaffold points at:
{
  "status": "completed",
  "bundle": "idea9_arm2/idea9_arm2_evaluation_result.json",
  "verdict": "NO CREDIT",
  "seeds_per_rung": 3,
  "rho_D0_mean": 0.46196451783180237,
  "rho_E1_mean": 0.05880538125832876,
  "improvement": 0.40315913657347363,
  "D0_seed_spread": 0.05683479830992737,
  "credit_rule": {
    "condition_1_exceeds_seed_spread": true,
    "condition_2_paired_bootstrap": true,
    "condition_3_no_guardrail_regression": false,
    "al

## Summary: what notebook 09b established, and what has superseded it

**What was established.** The ablation-ladder machinery for arm 2 works end to end. Rungs train, each rung
gets its own fingerprint so it can never be confused with the baseline, the collapse monitors report, the
shared arm 1 instrument re-scores every checkpoint on one ruler, and the credit rule executes. That is a
plumbing result.

**What was NOT established, and must not be quoted as if it were.** Anything about any real encoder. Every
number this notebook computes comes from a hand-authored synthetic cohort with a planted signal, trained at
smoke scale (32 frames, embed_dim 32, encoder_depth 1, 2 stage-0 epochs) on a model whose mean pairwise
cosine of about 0.9995 shows it is near collapse. In particular **D0 mean R-squared -0.320 with standard
deviation 0.036, E1 mean -0.294, effect 0.025, credited false** is PLUMBING VALIDATION ONLY. The
`artifact mode: real` line in several outputs refers only to the directory the bundle is written to.

**What is superseded, and where the correction lives.**

1. *The loss form.* The absolute equivariance term
   `L = mean(( s(enc(Mx)) + s(enc(x)) )^2)` with a trainable head `s` is SCALE-DEGENERATE: it can be
   satisfied by shrinking the head's output rather than by changing the encoder. On synthetic fixtures the
   term fell by a factor of 184.16 while the head's output scale shrank by a factor of 4.77, and rho moved
   by only 0.0096 against a gate of 0.0492 (0 is mirror equivariant, 4 is mirror blind). The repair is a
   scale-invariant ratio, and the real run used the `parameter_free` form of it. See section 2 and section
   8a.
2. *The head-only variant.* Penalising a swap of the head's own tokens is an algebraic no-op with exactly
   zero loss and exactly zero gradient. It trains nothing.
3. *The recipe.* Section 8's instruction to paste the absolute term into notebook 04 is marked
   `SUPERSEDED, DO NOT FOLLOW AS WRITTEN` and is kept only so the mistake stays visible and explained.

**The real arm 2 result.** `new_nb_09_00` through `new_nb_09_03`, running 3 seeds against 5 registered.
Verdict **NO CREDIT**: rho on the target encoder moved from a D0 mean of 0.462 to an E1 mean of 0.0588, an
improvement of 0.403 against a D0 seed spread of 0.0568, improving on 18 of 18 source videos, but the
`feature_std` guardrail regressed and variance removal is therefore a competing explanation for the gain.
The source-grouped five-class guardrail was NOT EVALUABLE.

**Next.** `new_nb_09_00_methodology_and_contract` to start arm 2 properly. `nb_09a` for arm 1's real
result on the frozen encoder, whose verdict is `ARTIFACT (side-agnostic nuisance control fired)`.
`nb_09c` for the possible-futures rehearsal and the external reach scaffold.

**Standing caveats.** All results transductive; the source video is the independent unit of evidence;
folder labels are dataset annotations, not clinical diagnoses; nothing here is clinical validation.
